In [ ]:
import re
import time
import ollama

# Configure your local Qwen model identifier
OLLAMA_MODEL_NAME = "gemma3:4b-it-qat"


def is_structural_table(text: str) -> bool:
  """Deterministic detector for tabular fragments (handles markdown and Docling formats)."""
  # Check for standard markdown table syntax
  has_markdown_table = text.count("|") >= 4 and (
      (":---" in text) or ("---:" in text)
  )
  
  return has_markdown_table


def predict_dynamic_k(
    chunk_text: str, model_name: str = OLLAMA_MODEL_NAME
) -> int:
  """Evaluates chunk semantic scope across k in {0, 1, 2, 3} using full-chunk analysis."""
  # ==========================================================================
  # Tier 1: Deterministic Syntax Pre-Gating (Host CPU, < 0.01 ms)
  # ==========================================================================
  if is_structural_table(chunk_text):
    return 3

  # ==========================================================================
  # Tier 2: Holistic Semantic & Epistemic Scope Classifier
  # ==========================================================================
  system_prompt = (
      "You are an expert computational linguist evaluating text chunks for a"
      " localized RAG document slice architecture.\n"
      "Analyze the complete semantic scope, conceptual density, and context"
      " requirements of the provided text chunk.\n\n"
      "Rate the chunk on a scale of 0 to 2:\n"
      "0 (Macro-Synthesis): Broad high-level summary, executive abstract, or"
      " global conclusion. Synthesizes overall research goals, main results, or"
      " systemic takeaways. Requires ZERO surrounding context.\n"
      "1 (Modular Exposition): Independent narrative prose, background"
      " context, hardware setups, or modular definitions. The concepts are"
      " self-contained and easily understood in isolation. Requires MINIMAL"
      " surrounding context.\n"
      "2 (Analytical Deep-Dive): Procedural formulations, mathematical proofs,"
      " experimental discussions, or architectural bottleneck evaluations."
      " While grammatically complete, the reasoning builds directly on ongoing"
      " arguments, formulas, or metrics in the section. Requires MODERATE"
      " surrounding context.\n\n"
      "Instruction: Respond with ONLY the single digit: 0, 1, or 2."
  )

  # Full-length, academically rigorous few-shot exemplars
  few_shot_calibration = (
      "Text:\n"
      "State-of-the-art AI architectures like Retrieval-Augmented Generation"
      " (RAG) often rely on massive data-center infrastructure, creating"
      " barriers to privacy-centric applications. This study introduces a"
      " localized 'Document Slice' framework using a bounded sliding window"
      " instead of full documents. Evaluated on 250 complex queries, our"
      " method achieves a 2.5x preprocessing speedup and a 51% reduction in"
      " energy consumption on an 8GB VRAM GPU with statistically insignificant"
      " impact on retrieval recall.\n"
      "Rating: 0\n\n"
      "Text:\n"
      "For benchmarking, we utilized an Intel Core i7-14650HX CPU paired with"
      " a mobile NVIDIA GeForce RTX 4060 GPU equipped with 8GB GDDR6 VRAM. The"
      " retrieval pipeline integrates LanceDB as a serverless local vector"
      " store, and embedding representations were executed using the"
      " nomic-embed-text-v1.5 model through the Ollama runtime framework.\n"
      "Rating: 1\n\n"
      "Text:\n"
      "During evaluation of the full-context baseline, the Key-Value (KV) cache"
      " footprint vastly exceeded physical memory limits, forcing the runtime"
      " to trigger CUDA Unified Memory paging into host RAM. This induced"
      " extensive page faulting across the PCIe bus and saturated the memory"
      " controller, causing the GPU to enter prolonged I/O stall states rather"
      " than sustaining compute throughput.\n"
      "Rating: 2\n\n"
  )

  user_prompt = f"{few_shot_calibration}Text:\n{chunk_text.strip()}\n\nRating:"

  response = ollama.chat(
      model=model_name,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
      ],
      options={
          "temperature": 0.0,
          "num_predict": 1,  # Single forward-pass argmax
          "top_p": 1.0,
      },
  )
  raw_output = response["message"]["content"].strip()
  match = re.search(r"[0-2]", raw_output)
  return int(match.group(0)) if match else 1


# ==============================================================================
# Verbatim Test Suite from citds.tex / citds.pdf
# ==============================================================================
corpus_test_chunks = [
    # --- Category 0: Macro-Synthesis / Global Summaries ---
    {
        "section": "Abstract: Overview & Main Results",
        "expected_k": 0,
        "text": (
            "State-of-the-art AI architectures like"
            ' Retrieval-Augmented Generation (RAG) using LLMs often rely on "Red'
            ' AI" infrastructure, creating significant barriers to'
            " privacy-centric, localized applications. Anthropic’s Contextual"
            " Retrieval addresses chunk-level context loss but introduces"
            " linear computational scaling that exceeds the limits of"
            ' consumer-grade hardware. This study proposes a "Document Slice"'
            " architecture that optimizes context generation by utilizing a"
            " sliding window of neighboring chunks rather than the entire"
            " document. By constraining the context window to a fixed radius,"
            " we facilitate efficient RAG operations on a single GPU with 8GB"
            " of VRAM."
        ),
    },
    {
        "section": "Section IV: Concluding Synthesis",
        "expected_k": 0,
        "text": (
            "This study demonstrates that state-of-the-art contextual"
            " retrieval can be successfully adapted for resource-constrained"
            " environments. By replacing full-document context with a localized"
            " 'Document Slice' sliding window, we have developed an RAG"
            " architecture that fits within the 8GB of VRAM available on"
            " consumer-grade GPUs. The proposed method achieves a 2.5x speedup"
            " and a 51% reduction in energy consumption with a statistically"
            " insignificant impact on retrieval recall. These results have"
            " significant implications for the development of private, local"
            " research assistants and the broader adoption of 'Green AI'"
            " practices."
        ),
    },
    # --- Category 1: Modular Exposition & Infrastructure ---
    {
        "section": "Introduction: Landscape Framing",
        "expected_k": 1,
        "text": (
            "The contemporary technological landscape is dominated by Large"
            " Language Models (LLMs) that have fundamentally altered how we"
            " interact with information. However, a growing 'compute divide'"
            " threatens the democratization of these tools. Much of the"
            " current literature assumes the availability of data-center-grade"
            " infrastructure, a trend dubbed 'Red AI' that prioritizes raw"
            " performance over computational efficiency. This reliance on"
            " massive resources creates a substantial barrier for localized,"
            " privacy-centric applications, particularly for academic teams"
            " and individual researchers."
        ),
    },
    {
        "section": "Section II-E: Hardware Specification",
        "expected_k": 1,
        "text": (
            "For benchmarking, we utilized a GeForce RTX 4060 Mobile GPU (8GB"
            " VRAM) and an Intel Core i7-14650HX CPU. The RAG pipeline"
            " integrated LanceDB for vector storage, and the embedding model"
            " executed in the Ollama inference framework."
        ),
    },
    # --- Category 2: Analytical Deep-Dives & Procedural Mechanics ---
    {
        "section": "Section II-C: Sliding Window Logic",
        "expected_k": 2,
        "text": (
            "For a given chunk i, the document slice is constructed as a"
            " continuous segment from chunk (i - k) to (i + k). This ensures"
            " the target chunk is positioned centrally within the prompt,"
            " mitigating the primacy and recency biases inherent in LLMs. For"
            " document boundaries (start or end), the window is truncated on"
            " one side and extended on the other to maintain a consistent total"
            " token count."
        ),
    },
    {
        "section": "Section II-F: Custom Metric Derivation (MAR)",
        "expected_k": 2,
        "text": (
            "We employed four primary metrics: Recall@K, Mean Reciprocal Rank"
            " (MRR), Mean Average Rank (MAR, our custom metric), and"
            " preprocessing latency. MAR = (1 / |Q|) * sum_q ( (1 / |R_q cap"
            " G_q|) * sum_c rank(c) ) where Q is the set of all queries, R_q"
            " is the set of retrieved documents, and G_q denotes ground-truth"
            " relevant documents."
        ),
    },
    {
        "section": "Section III-E: Memory Profiling Analysis",
        "expected_k": 2,
        "text": (
            "During the evaluation of the Anthropic baseline, a critical"
            " hardware bottleneck was identified. While the quantized weights"
            " of the model natively fit within the 8GB VRAM limit, the"
            " Key-Value (KV) cache footprint for full-document context"
            " sequences vastly exceeded the remaining physical memory. To"
            " prevent Out-Of-Memory (OOM) failures, the inference backend"
            " utilized CUDA Unified Memory, dynamically offloading the KV"
            " cache overflow into the host system RAM."
        ),
    },
    # --- Category 3: Tabular Environments ---
    {
        "section": "Table I: Ablation Benchmark Matrix",
        "expected_k": 3,
        "text": (
            "| Method | Time | Recall@20 | MRR@20 | MAR@20 |\n"
            "| :--- | :--- | :--- | :--- | :--- |\n"
            "| Hybrid retrieval | 0 | 0.9393 | 0.7748 | 4.0363 |\n"
            "| doc slice k=0 | 7m28s | 0.9373 | 0.7861 | 3.8083 |\n"
            "| doc slice k=1 | 12m08s | 0.9433 | 0.7990 | 3.7260 |\n"
            "| doc slice k=2 | 17m28s | 0.9467 | 0.7680 | 3.8095 |\n"
            "| doc slice k=3 | 22m41s | 0.9453 | 0.7877 | 4.0200 |\n"
            "| doc slice k=4 | 28m11s | 0.9433 | 0.8933 | 3.9588 |\n"
            "| Anthropic | 71m34s | 0.9540 | 0.7863 | 3.9355 |"
        ),
    },
    {
        "section": "Table II: Comparative Baseline Summary",
        "expected_k": 3,
        "text": (
            "| Metric | Anthropic | Doc slice | Relative Change |\n"
            "| :--- | :--- | :--- | :--- | :--- |\n"
            "| Recall@20 | 95.4% | 94.5% | -0.9% |\n"
            "| MRR@20 | 0.786 | 0.77 | -2% |\n"
            "| MAR@20 | 3.93 | 4.01 | 1.8% |\n"
            "| Preprocessing Time | 71m34s | 29m09s | -60% |"
        ),
    },
]

# ==============================================================================
# Execution Benchmark
# ==============================================================================
if __name__ == "__main__":
  print(
      f"{'Chunk Origin':<42} | {'Target':<6} | {'Pred':<6} | {'Result':<6} |"
      f" {'Latency':<8}"
  )
  print("-" * 76)

  exact_matches = 0
  latencies = []

  for item in corpus_test_chunks:
    t0 = time.perf_counter()
    k_pred = predict_dynamic_k(item["text"])
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    latencies.append(elapsed_ms)

    k_gold = item["expected_k"]
    matched = k_pred == k_gold
    if matched:
      exact_matches += 1

    status = "PASS" if matched else "FAIL"
    print(
        f"{item['section']:<42} | k={k_gold:<4} | k={k_pred:<4} | {status:<6}"
        f" | {elapsed_ms:6.1f} ms"
    )

  total = len(corpus_test_chunks)
  print("-" * 76)
  print(
      f"Strict Exact Match Accuracy: {exact_matches}/{total}"
      f" ({exact_matches/total*100:.1f}%)"
  )
  print(f"Mean Decision Latency:     {sum(latencies)/total:.2f} ms per chunk")

Chunk Origin                               | Target | Pred   | Result | Latency 
----------------------------------------------------------------------------
Abstract: Overview & Main Results          | k=0    | k=1    | FAIL   |  207.0 ms
Section IV: Concluding Synthesis           | k=0    | k=0    | PASS   |  168.5 ms
Introduction: Landscape Framing            | k=1    | k=1    | PASS   |  159.8 ms
Section II-E: Hardware Specification       | k=1    | k=1    | PASS   |  150.3 ms
Section II-C: Sliding Window Logic         | k=2    | k=1    | FAIL   |  154.1 ms
Section II-F: Custom Metric Derivation (MAR) | k=2    | k=2    | PASS   |  159.2 ms
Section III-E: Memory Profiling Analysis   | k=2    | k=2    | PASS   |  159.0 ms
Table I: Ablation Benchmark Matrix         | k=3    | k=3    | PASS   |    0.0 ms
Table II: Comparative Baseline Summary     | k=3    | k=3    | PASS   |    0.0 ms
----------------------------------------------------------------------------
Strict Exact Match Accura

In [1]:
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

pdf_path = "input/A_Feature_Fusion_Based_Indicator_for_Training-Free_Neural_Architecture_Search.pdf"

converter = DocumentConverter()
result = converter.convert(pdf_path)

tokenizer = HuggingFaceTokenizer(
			tokenizer=AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5"),
			max_tokens=2000
		)

chunker = HybridChunker(
            tokenizer=tokenizer,
            merge_peers=True  # Optional, defaults to true
        )

chunks = list(chunker.chunk(dl_doc=result.document))
joined_chunks = "\n\n".join([chunk.text for chunk in chunks])

with open("deleteme.md", "w", encoding="utf-8") as file:
	file.write(joined_chunks)

print("Saved converted Markdown to deleteme.md")

/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Preset 'granite_vision_v4' already registered for ChartExtractionVlmEngineOptions
[INFO] 2026-09-22 00:06:27,746 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-22 00:06:27,748 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-09-22 00:06:27,755 [RapidOCR] download_file.py:60: File exists and is valid: /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-22 00:06:27,756 [RapidOCR] main.py:50: Using /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-22 00:06:2

Saved converted Markdown to deleteme.md


In [5]:
from docling_core.types.doc import DocItemLabel

# Deterministic AST inspection (Zero CPU overhead, zero false positives)
table_counter = 0
for chunk in chunks:
	has_table = any(item.label == DocItemLabel.TABLE for item in chunk.meta.doc_items)
	# print(f"Has Table: {has_table}")
	# print(f"\t Chunk headings: {chunk.meta.headings}")
	# if "To deliver an optimal Mixed Reality (MR) experience, wherein virtual " in chunk.text:
	# 	print(f"Chunk Text:\n{chunk.text}\n")
	if has_table:
		table_counter += 1
		print(f"Chunk Text:\n{chunk.text}\n")
print(f"Total tables found: {table_counter}")

Chunk Text:
We compare the performance of our FI and standalone indicators, i.e. CJ, NLR, CNNTK, OS on CIFAR-10 of NASBench-101 search space. Because we have four indicators, there are 11 combinations for our FI. FI2 means two indicators are used and so on. We use f CJ, OS g for FI2, f CJ, CNNTK, OS g for FI3, and f CJ, NLR, CNNTK, OS g for FI4. We refer the reader to the Ablation section for the performance of other combinations. The results are shown in Table 1.
As shown in Table 1, the proposed FI outperforms other indicators signi cantly in all performance assessment methods. Notably, our FI achieves 5% higher KROCC and SROCCthan the CJ-based single indicator [16]. Compared to NLR, CNNTK and OS, the proposed FI has around 24% and 30% higher KROCC and SROCC, respectively. Additionally, it is worth noting that increasing the number of indicator does not improve the performance on NAS-Bench-101. One possible reason for this is that there are only three operations (i.e., 3 3 convolutio

In [75]:
import json
import re

def is_serialized_docling_table(text: str) -> bool:
    """
    Detects whether a raw text chunk contains a Docling-serialized table,
    differentiating actual tabular data from narrative citations (e.g., 'Table 1 illustrates').
    """
    # 1. Primary Docling Signature: Repeated Key-Value assignments ("Header = Value")
    # Natural prose or isolated equations rarely have >= 4 spaced equals signs.
    if text.count(" = ") >= 4:
        return True

    # 2. Secondary Signature: Formal Caption Declaration + Numerical Density
    # Matches "TABLE 1. Title" or "Table 2: Metrics", but NOT "Table 1 shows"
    has_formal_caption = bool(re.search(r"\b(?:TABLE|Table)\s+\d+[\.:]\s+[A-Z]", text))
    
    if has_formal_caption:
        tokens = text.split()
        if not tokens:
            return False
        # Calculate numerical token ratio
        num_count = sum(1 for t in tokens if re.search(r"\d", t))
        numerical_ratio = num_count / len(tokens)
        
        # True tables have high numerical token concentration (typically > 8%)
        if numerical_ratio >= 0.08:
            return True

    return False


# ==============================================================================
# Verification on Provided JSON Dataset
# ==============================================================================
if __name__ == "__main__":
    with open("split_documents/A_Feature_Fusion_Based_Indicator_for_Training-Free_Neural_Architecture_Search.pdf.json", "r", encoding="utf-8") as f:
        chunks = json.load(f)

    print(f"Loaded {len(chunks)} chunks. Scanning for serialized tables...\n")
    
    for idx, c in enumerate(chunks, 1):
        detected = is_serialized_docling_table(c["text"])
        snippet = c["text"].replace("\n", " ")[:65]
        
        if detected:
            print(f"[MATCH] Chunk {idx:<2} (ID: {c['id'][:8]}...) -> DETECTED AS TABLE")
            print(f"        Snippet: \"{snippet}...\"\n")
        else:
            # Check if text mentions 'Table' to verify false-positive rejection
            if "table" in c["text"].lower():
                print(f"[REJECT] Chunk {idx:<2} (ID: {c['id'][:8]}...) -> Mentioned 'Table' but correctly classified as Prose")
                print(f"         Snippet: \"{snippet}...\"\n")

Loaded 26 chunks. Scanning for serialized tables...

[MATCH] Chunk 15 (ID: 28d263ab...) -> DETECTED AS TABLE
        Snippet: "We compare the performance of our FI and standalone indicators, i..."

[MATCH] Chunk 16 (ID: 2a0165a2...) -> DETECTED AS TABLE
        Snippet: "We further evaluate the performance of training-free indicators o..."

[MATCH] Chunk 17 (ID: fceabdef...) -> DETECTED AS TABLE
        Snippet: "We perform cross dataset tests to verify the generalizability of ..."

[REJECT] Chunk 20 (ID: 8c2ab019...) -> Mentioned 'Table' but correctly classified as Prose
         Snippet: "When we develop an indicator for training-free NAS framework, the..."

[MATCH] Chunk 21 (ID: d42f8336...) -> DETECTED AS TABLE
        Snippet: "We also investigate the performance of our FI by combining differ..."

[MATCH] Chunk 22 (ID: a4de586a...) -> DETECTED AS TABLE
        Snippet: "There can be a few factors that threaten the validity of this res..."



# One-shot rewritten for Docling Hybrid Chunker

In [12]:
import re
import time
import ollama

# Configure your local quantized model identifier
OLLAMA_MODEL_NAME = "gemma3:4b-it-qat"


def is_structural_table(text: str) -> bool:
    """
    Deterministic structural detector for Docling-processed documents.
    
    Bypasses standard Markdown pipe heuristics ('|' and ':---') in favor of:
    1. Docling linearized relational assignments (' = ' or '. = ')
    2. Formal IEEE table caption declarations ('TABLE X. Title')
    """
    # 1. High density of serialized key-value assignments
    assignment_count = text.count(" = ") + text.count(". = ")
    if assignment_count >= 4:
        return True

    # 2. Formal IEEE Table Caption regex (e.g., 'TABLE 1. The ecological...')
    # Rejects casual in-text citations like 'as seen in Table 1, we observe'
    has_formal_caption = bool(re.search(r"\b(?:TABLE|Table)\s+[0-9IVXLCDM]+[\.:]\s+[A-Z]", text))
    if has_formal_caption:
        return True

    return False


def predict_dynamic_k(chunk_text: str, model_name: str = OLLAMA_MODEL_NAME) -> int:
    """
    Evaluates chunk context requirements across k in {0, 1, 2, 3}.
    Tier 1: Deterministic CPU layout gating (< 0.05 ms).
    Tier 2: Single-token greedy argmax forward pass using gemma3:4b-it-qat.
    """
    # ==========================================================================
    # Tier 1: Deterministic Layout Pre-Gating (Host CPU)
    # ==========================================================================
    if is_structural_table(chunk_text):
        return 3

    # ==========================================================================
    # Tier 2: Holistic Semantic & Epistemic Scope Classifier (Local SLM)
    # ==========================================================================
    system_prompt = (
        "You are an expert computational linguist evaluating text chunks for a localized RAG document slice architecture.\n"
        "Analyze the complete semantic scope, conceptual density, and context requirements of the provided text chunk.\n\n"
        "Rate the chunk on a scale of 0 to 2:\n"
        "0 (Macro-Synthesis): Broad high-level summary, executive abstract, or global conclusion. Synthesizes overall research goals, main results, or systemic takeaways. Requires ZERO surrounding context.\n"
        "1 (Modular Exposition): Independent narrative prose, background literature, hardware setups, or modular definitions. The concepts are self-contained and easily understood in isolation. Requires MINIMAL surrounding context.\n"
        "2 (Analytical Deep-Dive): Procedural formulations, mathematical proofs, experimental discussions, or architectural bottleneck evaluations. While grammatically complete, the reasoning builds directly on ongoing arguments, formulas, or metrics in the section. Requires MODERATE surrounding context.\n\n"
        "Instruction: Respond with ONLY the single digit: 0, 1, or 2."
    )

    # In-context few-shot exemplars calibrated on academic layout and formula artifacts
    few_shot_calibration = (
        "Text:\n"
        "ABSTRACT To deliver an optimal Mixed Reality (MR) experience, wherein virtual elements and real-world objects "
        "are seamlessly merged, it is vital to ensure a consistent vergence-accommodation distance. This necessitates the "
        "advancement of technology to precisely estimate the user's gaze distance. Presently, various MR devices employ small "
        "eye-tracking cameras to capture both eyes and infer the gaze distance based on vergence angle data. However, this "
        "technique faces significant challenges, as it is highly sensitive to several human errors, such as strabismus, blinking, "
        "and fatigue of the eyes due to prolonged use. To address these issues, this paper introduces an innovative hybrid "
        "algorithm for estimating gaze distances. The proposed approach concurrently utilizes an eye camera and a depth camera "
        "to conduct parallel estimations: one based on the conventional vergence angle and the other on gaze-mapped depth "
        "information. The confidence of each method is then assessed and cross-referenced, and an adaptive weighted average is "
        "computed to derive a more precise and stable gaze distance estimation.\n"
        "Rating: 0\n\n"
        "Text:\n"
        "In this section, we explain a hybrid method for eye-gaze distance estimation. Note that we do not explain specifics of "
        "2D gaze and a gaze distance from vergence because we only applied one of the commercial eye trackers, Pupil-Labs [8] "
        "of which provides 2D gaze with an accuracy of 0 . 60 ◦ and a precision of 0 . 02 ◦ . It is compatible with Intel RealSense "
        "RGB-D camera [19] (Fig. 1(a)) as the scene camera, and consists of two near-infrared (NIR) eye cameras, where the 2D gaze "
        "is calculated by the intersection of the gaze vectors from each eye in the scene camera. We exploit the human visual "
        "perception mechanism [12] (Fig. 2(a)) within the eyegaze distance estimation framework by cross-referencing the gaze "
        "distance derived from vergence with that obtained from IEEE Access gaze-mapped depth as shown in Fig. 2(b).\n"
        "Rating: 1\n\n"
        "Text:\n"
        "Different face shapes of users (i.e. eye position, eye ball size, inter pupillary distance, etc.) can cause scale "
        "errors of gaze distance from vergence even though the 2D gaze point is well estimated. We exploit the gaze distance from "
        "gazemapped depth as a guided information to refine the scale of the initial gaze distance from vergence because the IR "
        "active sensor is relatively robust to human factors. Based on the characteristics of which the gaze distance from "
        "vergence changes roughly linearly to that of gaze-mapped depth as shown in Fig. 3, we exploit 1 st order polynomial function "
        "to refine the initial gaze distance from vergence.\n"
        "<!-- formula-not-decoded -->\n"
        "where ˆ V is the refined gaze distance from vergence for given initial gaze distance from vergence V and α 0 , α 1 are "
        "the polynomial coefficients. The optimal coefficients can be estimated by minimizing an objective function defined as\n"
        "<!-- formula-not-decoded -->\n"
        "where N is the number of samples, V n , and D n are n th initial gaze distance from vergence and corresponding that of "
        "gazemapped depth, respectively.\n"
        "Rating: 2\n\n"
    )

    user_prompt = f"{few_shot_calibration}Text:\n{chunk_text.strip()}\n\nRating:"

    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        options={
            "temperature": 0.0,
            "num_predict": 1,  # Single-token argmax termination
            "top_p": 1.0
        }
    )

    raw_output = response["message"]["content"].strip()
    match = re.search(r"[0-2]", raw_output)
    return int(match.group(0)) if match else 1


# ==============================================================================
# Verbatim Test Suite from The_Application_of_the_SOFM_Neural_Network_and_IoT...json
# ==============================================================================
corpus_test_chunks = [
    # --- Category 0: Macro-Synthesis / Global Summaries ---
    {
        "section": "Chunk 1: Abstract Overview & Empirical Zoning Results",
        "expected_k": 0,
        "text": (
            "ABSTRACT To promote rural revitalization under the premise of fully considering the ecological risk factors of "
            "land consolidation, this study takes A County in Shaanxi Province as a case study and introduces Self-Organizing "
            "Feature Map (SOFM) neural network and Internet of Things (IoT) technology to partition the land. In this study, a "
            "comprehensive index system is constructed based on IoT technology, and the relevant factors are quantitatively "
            "analyzed from the perspective of land consolidation ecological risk. Then, the land consolidation project area's "
            "attribute and geographic space domains are used as the input of the SOFM neural network to reveal the distribution "
            "and influence degree of each factor in the area and determine the zoning pattern of land consolidation in A County. "
            "The results show that the rural revitalization land zoning pattern of land consolidation in A County, Shaanxi Province, "
            "is divided into four land consolidation areas. Firstly, the priority remediation area soon covers 27 administrative "
            "villages with a total area of 28090 hm 2 , accounting for 32.75%. Secondly, the moderate renovation area is soon "
            "classified into 37 administrative villages, with a total area of 15986 hm 2 , equivalent to 18.55%. In the medium term, "
            "the land-saving renovation area covers 39 administrative villages with a total area of 19686 hm 2 , accounting for 22.75% "
            "of the total area. Finally, in the long-term restricted remediation area, 37 administrative villages are divided, "
            "with a total area of 22081 hm 2 , accounting for 25.67% of the total area. These data results provide a quantitative "
            "basis for land consolidation planning in this area to achieve the goal of rural land revitalization in different time ranges."
        )
    },
    {
        "section": "Chunk 14: Section V-B Concluding Synthesis",
        "expected_k": 0,
        "text": (
            "In conclusion, with the support of digital finance and IoT technology, this study has successfully utilized the "
            "SOFM neural network method to delineate rural revitalization land use in A County, Shaanxi Province. This study "
            "provides scientific support for the development of rural revitalization. Future research directions should focus "
            "on further investigating the role and impact of digital finance in rural revitalization. This involves examining "
            "the contributions of various digital finance tools to rural financial services, agricultural production, and rural "
            "economic development to formulate more specific policies and strategies."
        )
    },

    # --- Category 1: Modular Exposition & Infrastructure ---
    {
        "section": "Chunk 4: Section II Related Works Literature Review",
        "expected_k": 1,
        "text": (
            "Many researchers and scholars have discussed this. Kusadokoro and Chitose used county panel data to evaluate the impact "
            "of road infrastructure development in Inner Mongolia in China on regional economic growth and urban-rural income inequality "
            "from 1999 to 2018. The results showed that the number of road infrastructure in Inner Mongolia had a strong positive impact "
            "on economic growth, but a strong negative impact on urban-rural income inequality at the county level [4]. Qu et al. employed "
            "large-scale remote sensing data, a topographic fluctuation range model, and a spatial econometric model to analyze the "
            "multifunctional change and its dynamic mechanism of gully agriculture in the Loess Plateau to understand the internal meaning "
            "of evolution and differentiation at the basin level. The results indicated that many key policies in the Loess Plateau would "
            "directly affect the evolution path of functions, thus providing policy ideas for the high-quality development of agriculture in the Loess Plateau [5]."
        )
    },
    {
        "section": "Chunk 8: Section IV-A Baseline Data Acquisition",
        "expected_k": 1,
        "text": (
            "The acquisition of the required basic data mainly covers the following aspects: 1 It covers fundamental land use data, "
            "comprising surveys of land use status, data concerning land use changes, agricultural land zoning data, and comprehensive "
            "land use planning. 2 It includes data on natural resources and environmental conditions, including terrain slope, elevation, "
            "geological conditions, soil characteristics, hydrological information, etc. 3 It spans socio-economic data, including administrative "
            "divisions, population statistics, regional gross output value, and road and traffic distribution.\n"
            "Amongtheabove data, the basic land use data mainly come from the database of land use change (2022 edition) and agricultural "
            "land quality classification (2021 edition) in A County of Shaanxi Province. The data on natural resources and environmental "
            "conditions are mainly obtained from the image map of A County in Shaanxi Province, the 1: 10000 digital elevation mode (DEM) "
            "data of A County in Shaanxi Province, and the statistical yearbook of A County in Shaanxi Province in 2022."
        )
    },
    {
        "section": "Chunk 10: Section IV-C Parameters & Weight Setting",
        "expected_k": 1,
        "text": (
            "SOFM neural network parameters: the number of iteration times, input, and output nodes of the SOFM neural network are set. "
            "The SOFM neural network can effectively divide A County in Shaanxi Province into land consolidation project areas with "
            "similar attributes and adjacent spaces through the input data of the attribute and geographic space domains. Wherein the "
            "number of input and output nodes is 10 and 30. The number of iterations is 1000.\n"
            "Weight of comprehensive index system: According to the research objectives and case characteristics, the weights of "
            "different comprehensive indexes are set to analyze the related factors of land consolidation projects quantitatively. The "
            "setting of these weights will affect the final rural revitalization land zoning pattern. Among them, the ecological risk "
            "weight of land consolidation is 0.4. The time urgency weight is 0.3. The spatial suitability weight is 0.3."
        )
    },

    # --- Category 2: Analytical Deep-Dives & Procedural Formulations ---
    {
        "section": "Chunk 6: Section III-A SOFM Distance & Weight Updates",
        "expected_k": 2,
        "text": (
            "Firstly, the SOFM neural network model is established, the parameter values are set, and then the coordinates of the "
            "input data set and the index values of various attribute spaces are dimensionless [22], [23], [24]. Subsequently, this "
            "study uses the method of mixed distance to describe the similarity between sampling points [25], [26], [27]. A point set has "
            "dual attributes of time and space, and its definition is as follows :\n"
            "<!-- formula-not-decoded -->\n"
            "<!-- formula-not-decoded -->\n"
            "{ g 1 N , g 2 N , . . . , g G N } indicates geographical space, and G = 1 , 2 , 3. { a 1 N , a 2 N , . . . , a D N } "
            "signifies the attribute space; D denotes the number of attributes; D s ij refers to the size of the geographical space "
            "between two points. wd stands for the weight of attribute d . ∑ wd = 1. a d i and a d j represent the value of attribute d "
            "in point i and the value of attribute d in point j . ws means the weight of geographical space. wa indicates the weight of "
            "attribute space, ws + wa = 1.\n"
            "In the SOFM neural network, the updating rule of neuron weight can be expressed as :\n"
            "<!-- formula-not-decoded -->\n"
            "1 wji means the weight update, α ( t ) is the learning rate, hij indicates the proximity function, xi refers to the "
            "input data; wji represents the neuron weight [28], [29], [30], [31]."
        )
    },
    {
        "section": "Chunk 7: Section III-B Relative Risk Community Formulation",
        "expected_k": 2,
        "text": (
            "The existing land consolidation projects in A County of Shaanxi Province are distributed in various towns and villages in "
            "this area, involving many factors, and there are complex relationships among them [38]. By constructing a relative risk "
            "model, the results of exposure-hazard analysis are characterized, and the relative risk value of each risk community is "
            "calculated. The calculation process reads :\n"
            "<!-- formula-not-decoded -->\n"
            "RSi represents the relative risk value of the i th risk community. j is the source of risk. k means the habitat type. "
            "m refers to the ecological receptor type. Sij signifies the density of risk sources. Hik indicates the habitat abundance. "
            "Xjk is the exposure coefficient. Ekm expresses the response coefficient [39], [40]. The density of risk sources is the "
            "ratio of the area of a risk source in the risk community to the maximum area of this risk source in the risk community. "
            "Habitat abundance is the ratio of a habitat area in the risk community to the maximum value of this habitat area in the "
            "risk community. The exposure coefficient is the ratio of the area of a risk source in the habitat to the total area of the habitat [41], [42], [43]."
        )
    },
    {
        "section": "Chunk 12: Section IV-E Discussion on Ecological Receptor Ranges",
        "expected_k": 2,
        "text": (
            "The ecological risks of landscape pattern (the ecological risk value is between 0.01 and 1.62) and soil (the ecological "
            "risk value is between 0.01 and 1.46) in A County of Shaanxi Province are markedly higher than those of the other three "
            "types of ecological receptors. This that improper measures during the land consolidation project's execution may lead to "
            "landscape fragmentation and soil degradation in the area.In contrast, the ecological risks of the water environment "
            "(ranging from 0.01 to 1.08) and biodiversity (ranging from 0.01 to 1.04) exhibit slightly lower values. This is consistent "
            "with that A County in Shaanxi Province is situated in the southeast paddy field agricultural area, with developed irrigation "
            "conditions and rich biological species characteristics in the subtropical monsoon climate zone."
        )
    },

    # --- Category 3: Tabular Environments ---
    {
        "section": "Chunk 11: Section IV-D Table 1 Bisection & Caption",
        "expected_k": 3,
        "text": (
            "The ecological risk analysis results of different ecological receptors are exhibited in Table 1.\n"
            "The above data results demonstrate differences among different areas regardingsoil ecological risk, water environment "
            "ecological risk, biodiversity ecological risk, and landscape pattern ecological risk. Soil ecological risk values reflect\n"
            "TABLE 1. The ecological risk analysis results of ecological receptors.\n"
            "soil fertility, pollution levels, and sustainability for land use. Lower soil ecological risk values indicate better "
            "suitability for agricultural production and resource utilization, which is crucial for rural economic development in the "
            "context of rural revitalization. The extremely low soil ecological risk values in Area C suggest that this area holds "
            "potential advantages for agricultural development."
        )
    }
]


# ==============================================================================
# Execution Benchmark Harness
# ==============================================================================
if __name__ == "__main__":
    print(f"{'Chunk Section Identifier':<46} | {'Gold':<5} | {'Pred':<5} | {'Result':<6} | {'Latency':<8}")
    print("-" * 80)

    exact_matches = 0
    latencies = []

    for item in corpus_test_chunks:
        t0 = time.perf_counter()
        k_pred = predict_dynamic_k(item["text"])
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        latencies.append(elapsed_ms)

        k_gold = item["expected_k"]
        matched = (k_pred == k_gold)
        if matched:
            exact_matches += 1

        status = "PASS" if matched else "FAIL"
        print(f"{item['section'][:46]} | k={k_gold:<3} | k={k_pred:<3} | {status:<6} | {elapsed_ms:6.1f} ms")

    total = len(corpus_test_chunks)
    print("-" * 80)
    print(f"Strict Exact Match Accuracy: {exact_matches}/{total} ({exact_matches/total*100:.1f}%)")
    print(f"Mean Decision Latency:     {sum(latencies)/total:.2f} ms per chunk")

Chunk Section Identifier                       | Gold  | Pred  | Result | Latency 
--------------------------------------------------------------------------------
Chunk 1: Abstract Overview & Empirical Zoning  | k=0   | k=1   | FAIL   |  233.9 ms
Chunk 14: Section V-B Concluding Synthesis | k=0   | k=0   | PASS   |  145.6 ms
Chunk 4: Section II Related Works Literature R | k=1   | k=0   | FAIL   |  155.1 ms
Chunk 8: Section IV-A Baseline Data Acquisitio | k=1   | k=1   | PASS   |  159.0 ms
Chunk 10: Section IV-C Parameters & Weight Set | k=1   | k=1   | PASS   |  154.3 ms
Chunk 6: Section III-A SOFM Distance & Weight  | k=2   | k=1   | FAIL   |  196.7 ms
Chunk 7: Section III-B Relative Risk Community | k=2   | k=1   | FAIL   |  168.6 ms
Chunk 12: Section IV-E Discussion on Ecologica | k=2   | k=0   | FAIL   |  150.6 ms
Chunk 11: Section IV-D Table 1 Bisection & Cap | k=3   | k=3   | PASS   |    0.0 ms
--------------------------------------------------------------------------------
Str